# Metadata Query Agent Evaluation with Strands Framework

This notebook evaluates the Metadata Query Agent using the AWS Strands evaluation framework.

## AWS Credentials Setup

The metadata query agent needs AWS credentials to access:
- **Amazon Bedrock** - For Claude model inference (`global.anthropic.claude-sonnet-4-6`)
- **Amazon Bedrock Knowledge Base** - For semantic context retrieval (`retrieve_kb_context`)
- **Amazon Athena** - For executing SQL queries (`execute_sql_query`)
- **Amazon SSM Parameter Store** - For retrieving the Athena results bucket name
- **Amazon DynamoDB** - For reading ontology metadata by `id`

### Credential Injection

This notebook uses an AWS profile configured locally. The boto3 session is created with this profile and then injected into the metadata query agent using `set_boto_session()`. This ensures all AWS service calls within the agent use the same credentials.

### Agent Architecture

The metadata query agent converts natural language questions to SQL using metadata from the Bedrock Knowledge Base:

1. **`retrieve_kb_context(user_query)`** — queries the Bedrock KB for table/column metadata, business term aliases, and database/catalog routing information
2. **`disambiguate_query_terms(user_query, kb_context)`** — maps query terms to specific tables using KB chunk metadata; returns CLEAR, AMBIGUOUS, or UNKNOWN status
3. **`execute_sql_query(sql_query, database_name, catalog_id)`** — runs the generated SQL on Athena and returns structured results

### Required Environment Variables
- `SEMANTIC_RAG_KB_ID` — Bedrock Knowledge Base ID for metadata retrieval
- `ONTOLOGY_METADATA_TABLE` — DynamoDB table name (default: `semantic-layer-metadata`)
- `ATHENA_WORKGROUP` — Athena workgroup for query execution
- `PROJECT_NAME` or `ATHENA_RESULTS_BUCKET` — S3 bucket for Athena query results
- `AWS_REGION` — AWS region (default: `us-east-1`)
- `EVAL_METADATA_QUERY_ID` — DynamoDB item `id` to use for evaluation runs

In [1]:
# Install required packages for evaluation
# !pip install strands-agents strands-agents-tools strands-agents-evals python-dotenv pandas --quiet

In [2]:
# Import required libraries
import sys
import os
import json
import pandas as pd
import boto3
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv

# Add agents directory to path
sys.path.append('../agents')

# Set AWS profile BEFORE importing the agent module so module-level boto3 clients
# (e.g. the DynamoDB resource in metadata_query_agent.main) pick up the right credentials.
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac-demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

# Import agent functions
from metadata_query_agent.main import invoke, reset_agent_state, set_boto_session

# Verify strands packages
try:
    import strands
    import strands_evals
    print("✓ strands-agents installed:", strands.__version__ if hasattr(strands, '__version__') else "version unknown")
    print("✓ strands-evals installed:", strands_evals.__version__ if hasattr(strands_evals, '__version__') else "version unknown")
except ImportError as e:
    print(f"✗ Import error: {e}")

print("✓ Imports successful")

✓ strands-agents installed: version unknown
✓ strands-evals installed: version unknown
✓ Imports successful


In [3]:
# Configure environment variables
os.environ['AWS_PROFILE']        = os.environ.get('AWS_PROFILE', 'huthmac-demo')
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')
os.environ['SEMANTIC_RAG_KB_ID']      = os.environ.get('SEMANTIC_RAG_KB_ID', 'XXX')
os.environ['ONTOLOGY_METADATA_TABLE'] = os.environ.get('ONTOLOGY_METADATA_TABLE', 'XXX')
os.environ['ATHENA_WORKGROUP']        = os.environ.get('ATHENA_WORKGROUP', 'XXX')
os.environ['PROJECT_NAME']            = os.environ.get('PROJECT_NAME', 'semantic-layer')

# S3 Tables (Iceberg) data source config — loaded from .env
# Same pattern as metadata_agent_evaluation.ipynb
S3T_CATALOG  = os.environ.get('S3T_CATALOG', 'XXX')
S3T_DATABASE = os.environ.get('S3T_DATABASE', 'XXX')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
try:
    identity = sts.get_caller_identity()
    print(f"✓ AWS Credentials Verified:")
    print(f"  Profile: {os.environ['AWS_PROFILE']}")
    print(f"  Account: ...{identity['Account'][-4:]}")
    print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
    print(f"  Region: {region}")
except Exception as e:
    print(f"✗ Failed to verify AWS credentials: {str(e)}")
    raise

# Inject session into agent
set_boto_session(session)
print(f"\n✓ Agent configured to use boto3 session credentials")
print(f"  All AWS service calls (Bedrock, Bedrock KB, Athena, SSM, DynamoDB) will use this session")

print(f"\n✓ Environment variables:")
for key in ['AWS_PROFILE', 'AWS_DEFAULT_REGION', 'SEMANTIC_RAG_KB_ID',
            'ONTOLOGY_METADATA_TABLE', 'ATHENA_WORKGROUP', 'PROJECT_NAME']:
    print(f"  {key} = {os.environ.get(key, '(not set)')}")

print(f"\n✓ Catalog/database configured:")
print(f"  S3T_CATALOG  = {S3T_CATALOG}")
print(f"  S3T_DATABASE = {S3T_DATABASE}")

# Load metadata_id stored by metadata_agent_evaluation.ipynb via %store
%store -r metadata_id
EVAL_ID = metadata_id
print(f"\n✓ Loaded EVAL_ID from %store: {EVAL_ID}")
print(f"  EVAL_METADATA_QUERY_ID = ...{EVAL_ID[-4:]}")

2026-03-22 16:48:22,392 - metadata_query_agent.main - INFO - Boto3 session set with region: us-east-1


✓ AWS Credentials Verified:
  Profile: huthmac-demo
  Account: ...4087
  User/Role: huthmac-Isengard
  Region: us-east-1

✓ Agent configured to use boto3 session credentials
  All AWS service calls (Bedrock, Bedrock KB, Athena, SSM, DynamoDB) will use this session

✓ Environment variables:
  AWS_PROFILE = huthmac-demo
  AWS_DEFAULT_REGION = us-east-1
  SEMANTIC_RAG_KB_ID = CUCH5JMBVX
  ONTOLOGY_METADATA_TABLE = semantic-layer-metadata
  ATHENA_WORKGROUP = semantic-layer-workgroup
  PROJECT_NAME = semantic-layer

✓ Catalog/database configured:
  S3T_CATALOG  = s3tablescatalog/semantic-layer-analytics-tables
  S3T_DATABASE = normalized

✓ Loaded EVAL_ID from %store: semantic-rag-single_table_basic-8f7bcf77
  EVAL_METADATA_QUERY_ID = ...cf77


## Load Evaluation Dataset

In [4]:
# Load evaluation dataset
with open('../data/eval/metadata_query_agent_evaluation_dataset.json', 'r') as f:
    test_cases = json.load(f)

print(f"Loaded {len(test_cases)} test cases")
print(f"\nCategories: {set(tc['category'] for tc in test_cases)}")
print(f"\nTest cases:")
for tc in test_cases:
    print(f"  [{tc['id']}] ({tc['category']}): {tc['query'][:60]}")

Loaded 1 test cases

Categories: {'admin_codes_test1'}

Test cases:
  [test1] (admin_codes_test1): How many unique admincodes exist?


## Setup Strands Evaluation Framework

Configure telemetry and evaluation infrastructure for capturing agent execution traces.

In [5]:
from strands_evals import Case, Experiment
from strands_evals.evaluators import (
    HelpfulnessEvaluator,
    FaithfulnessEvaluator,
    ToolSelectionAccuracyEvaluator,
    ToolParameterAccuracyEvaluator,
    GoalSuccessRateEvaluator,
)
from strands_evals.mappers import StrandsInMemorySessionMapper
from strands_evals.telemetry import StrandsEvalsTelemetry

# Setup telemetry for capturing agent spans
telemetry = StrandsEvalsTelemetry().setup_in_memory_exporter()
memory_exporter = telemetry.in_memory_exporter

print("✓ Strands evaluation framework initialized")
print(f"  Telemetry: In-memory exporter configured")
print(f"  Ready to capture agent traces")

2026-03-22 16:48:22,426 - strands_evals.telemetry.config - INFO - Initializing tracer for strands-evals
2026-03-22 16:48:22,427 - strands_evals.telemetry.config - INFO - Enabling in-memory export for strands-evals


✓ Strands evaluation framework initialized
  Telemetry: In-memory exporter configured
  Ready to capture agent traces


In [6]:
# Convert test cases to Strands Case objects
strands_cases = []

for test_case in test_cases:
    case = Case[str, str](
        name=test_case['id'],
        input=test_case['query'],
        expected_output=test_case['expected'],
        metadata={
            'category': test_case['category'],
            'id': EVAL_ID,
            'expected_tools': test_case.get('expected_tools', []),
            'expected_tables': test_case.get('expected_tables', []),
            'catalog_id': S3T_CATALOG,
            'database': S3T_DATABASE,
        }
    )
    strands_cases.append(case)

print(f"✓ Converted {len(strands_cases)} test cases to Strands Case format")
print(f"\nCases:")
categories = {}
for case in strands_cases:
    cat = case.metadata['category']
    categories[cat] = categories.get(cat, 0) + 1
    print(f"  [{case.name}] ({cat}): {case.input[:60]}")

print(f"\nCategories: {categories}")

✓ Converted 1 test cases to Strands Case format

Cases:
  [test1] (admin_codes_test1): How many unique admincodes exist?

Categories: {'admin_codes_test1': 1}


## Define Task Function

Create the task function that invokes the metadata query agent and captures execution traces for evaluation.

The `invoke()` payload requires:
- `question`: the natural language query
- `id`: the DynamoDB item ID — the agent looks this up to confirm the config exists before running

In [7]:
def metadata_query_agent_task(case: Case) -> dict:
    """
    Task function that executes the metadata query agent and captures telemetry.

    1. Clears previous telemetry spans
    2. Resets agent state for a clean run
    3. Calls invoke() with {question, id}
    4. Captures execution spans and maps to a session for trajectory evaluation

    Returns:
        dict: 'output' (agent answer), 'trajectory' (session spans), 'duration', 'success', 'error'
    """
    memory_exporter.clear()
    reset_agent_state()

    try:
        start_time = datetime.now()
        payload = {
            "question": case.input,
            "id": case.metadata.get('id', EVAL_ID),
        }
        response = invoke(payload=payload, context={})
        duration = (datetime.now() - start_time).total_seconds()

        # Extract answer text from response dict
        if isinstance(response, dict):
            response_text = response.get('answer', response.get('result', response.get('error', str(response))))
        else:
            response_text = str(response)

        # Capture spans and map to session
        finished_spans = list(memory_exporter.get_finished_spans())
        mapper = StrandsInMemorySessionMapper()
        session_obj = mapper.map_to_session(finished_spans, session_id=case.session_id)

        return {
            "output": response_text,
            "trajectory": session_obj,
            "duration": duration,
            "success": True,
            "error": None,
        }

    except Exception as e:
        print(f"✗ Error in {case.name}: {str(e)}")
        import traceback; traceback.print_exc()
        return {"output": "", "trajectory": None, "duration": 0, "success": False, "error": str(e)}

print("✓ Task function defined")
print(f"  Payload: 'question': ..., 'id': ...{EVAL_ID[-4:]}")
print("  Captures: Agent answer + execution traces")

✓ Task function defined
  Payload: 'question': ..., 'id': ...cf77
  Captures: Agent answer + execution traces


## Create Built-in Evaluators

Configure Strands evaluators for multi-dimensional assessment of query quality.

In [8]:
from strands.models import BedrockModel

# Judge model — same boto session, zero temperature for deterministic scoring
judge_model = BedrockModel(
    model_id='global.anthropic.claude-sonnet-4-6',
    temperature=0.0,
    boto_session=session,
)

# ── Custom system prompt: ToolParameterAccuracyEvaluator ────────────────────
# Framework limitation: _extract_tool_level() builds session_history BEFORE
# iterating tool_spans in the same trace, so tool #3 (execute_sql_query)
# cannot see the outputs of tool #1 (retrieve_kb_context) or tool #2
# (disambiguate_query_terms) — they all share the same pre-trace snapshot.
#
# Workaround: tell the judge that parameters consistent with the expected
# workflow are grounded, even when prior tool outputs are absent from the
# visible conversation history.
TOOL_PARAM_SYSTEM_PROMPT = """You are an objective judge evaluating whether the parameter values in an AI assistant's tool call are grounded in the available context.

## Expected Workflow (parameter provenance reference)
The metadata query agent follows this three-step workflow within a SINGLE agent turn:
1. `retrieve_kb_context(user_query)` — retrieves table/column metadata, database_name, and catalog_id from the Bedrock Knowledge Base
2. `disambiguate_query_terms(user_query)` — resolves natural language terms to specific table names using KB chunk metadata
3. `execute_sql_query(sql_query, database_name, catalog_id)` — executes SQL on Athena using the database and catalog discovered in steps 1-2

IMPORTANT: Because all three tool calls occur in the same agent turn, the evaluation
framework may NOT include prior tool outputs in the conversation history shown to you.
This is a known framework limitation — the absence of prior tool outputs does NOT mean
the values were fabricated. The agent has access to all prior tool results at runtime.

## Grounding Sources
Parameter values are considered grounded (NOT fabricated) if they come from ANY of:
1. The user's original message (e.g. the natural language query)
2. The output/results of a previously executed tool call in the same turn (even if not shown)
3. Reasonable inference from the workflow (e.g. constructing SQL from KB-retrieved schema)

## Evaluation Guidelines:
- For `retrieve_kb_context` and `disambiguate_query_terms`: the `user_query` parameter should match the user's message
- For `execute_sql_query`: `database_name`, `catalog_id`, table names, and column names are expected to come from the KB context retrieved in step 1. These are grounded even if the KB output is not visible in the conversation history. Do NOT penalise these values.
- Only flag parameters that contradict the user's intent or are clearly nonsensical

## Output Format:
First, provide a brief analysis of each parameter.
Then, answer with EXACTLY ONE of these responses:
- Yes (if parameter values are consistent with the user's question and the expected workflow)
- No (if any parameter value contradicts the user's intent or is clearly nonsensical)"""

# ── Custom system prompt: GoalSuccessRateEvaluator ──────────────────────────
# Framework limitation: the Strands structured_output mechanism injects an
# internal "user" message asking the agent to format its response as structured
# output. _extract_session_level() picks this up as a real user goal, causing
# the judge to evaluate "did the agent format as structured output?" instead of
# "did the agent answer the user's data question?".
#
# Workaround: tell the judge what the real user goals are and to ignore any
# internal framework messages about structured output formatting.
GOAL_SUCCESS_SYSTEM_PROMPT = """You are an objective judge evaluating whether an AI assistant successfully completed the user's goal in a conversation.

## Context
The AI assistant is a metadata query agent that answers natural language questions about data by:
1. Retrieving metadata from a Knowledge Base (`retrieve_kb_context`)
2. Disambiguating query terms (`disambiguate_query_terms`)
3. Executing SQL queries on Athena (`execute_sql_query`)

## Identifying the User's Goal
The user's goal is ALWAYS the first user message in the conversation — a natural language
question about data (e.g. "How many unique admincodes exist?"). This is the ONLY goal
to evaluate.

IMPORTANT: Ignore any subsequent "user" messages that ask about formatting, structured
output, or response format. These are internal framework messages, NOT real user goals.
Do NOT evaluate the agent against these messages.

## Evaluation Criteria
The agent successfully achieved the goal if it:
1. Retrieved relevant KB context for the question
2. Identified the correct table(s) and column(s)
3. Constructed a reasonable SQL query that would answer the question
4. Attempted to execute the query

If the SQL execution fails due to infrastructure issues (permissions, timeouts, etc.)
that are outside the agent's control, the goal is STILL considered achieved as long as
the agent took the correct steps (1-3) and communicated the issue clearly to the user.

## Output Format:
First, identify the user's data question from the first message.
Then, evaluate whether the agent took the correct steps to answer it.
Finally, answer with EXACTLY ONE of these responses:
- Yes (agent identified correct tables/columns, built valid SQL, and either returned results or clearly communicated an infrastructure error)
- No (agent failed to identify the right data, built incorrect SQL, or did not attempt to answer the question)"""

evaluators = [
    # TRACE_LEVEL: Was the answer helpful to the user?
    HelpfulnessEvaluator(model=judge_model),

    # TRACE_LEVEL: Did the answer faithfully reflect the SQL results (no hallucinations)?
    FaithfulnessEvaluator(model=judge_model),

    # TOOL_LEVEL: Did the agent call retrieve_kb_context → disambiguate → execute_sql in order?
    ToolSelectionAccuracyEvaluator(model=judge_model),

    # TOOL_LEVEL: Were the SQL query, database_name, and catalog_id parameters correct?
    ToolParameterAccuracyEvaluator(model=judge_model, system_prompt=TOOL_PARAM_SYSTEM_PROMPT),

    # SESSION_LEVEL: Did the agent successfully answer the user's question end-to-end?
    GoalSuccessRateEvaluator(model=judge_model, system_prompt=GOAL_SUCCESS_SYSTEM_PROMPT),
]

print(f"✓ Created {len(evaluators)} evaluators:")
print(f"  1. HelpfulnessEvaluator           - Answer quality & user satisfaction")
print(f"  2. FaithfulnessEvaluator          - Groundedness & hallucination detection")
print(f"  3. ToolSelectionAccuracyEvaluator - Correct tool choice & ordering")
print(f"  4. ToolParameterAccuracyEvaluator - SQL query & parameter validation (custom prompt: same-turn grounding)")
print(f"  5. GoalSuccessRateEvaluator       - End-to-end goal achievement (custom prompt: ignore structured output messages)")
print(f"\n  All evaluators use claude-sonnet-4-6 as judge")

✓ Created 5 evaluators:
  1. HelpfulnessEvaluator           - Answer quality & user satisfaction
  2. FaithfulnessEvaluator          - Groundedness & hallucination detection
  3. ToolSelectionAccuracyEvaluator - Correct tool choice & ordering
  4. ToolParameterAccuracyEvaluator - SQL query & parameter validation (custom prompt: same-turn grounding)
  5. GoalSuccessRateEvaluator       - End-to-end goal achievement (custom prompt: ignore structured output messages)

  All evaluators use claude-sonnet-4-6 as judge


## Run Evaluation Experiment

Execute test cases and evaluate with all built-in evaluators.

In [9]:
experiment = Experiment[str, str](
    cases=strands_cases,
    evaluators=evaluators,
)

print(f"Starting experiment with {len(strands_cases)} test case(s)...")
print(f"{'='*70}\n")

reports = experiment.run_evaluations(metadata_query_agent_task)

print(f"\n{'='*70}")
print(f"✓ Evaluation complete!")
print(f"  Generated {len(reports)} evaluation report(s)")

2026-03-22 16:48:22,487 - metadata_query_agent.main - INFO - Agent state reset for session: None
2026-03-22 16:48:22,488 - metadata_query_agent.main - INFO - Agent state reset for session: 5fe452f0


Starting experiment with 1 test case(s)...



2026-03-22 16:48:22,672 - metadata_query_agent.main - INFO - Metadata Query Agent created with Bedrock KB integration
2026-03-22 16:48:22,673 - metadata_query_agent.main - INFO - Session 5fe452f0 — id=semantic-rag-single_table_basic-8f7bcf77 q=How many unique admincodes exist?...
2026-03-22 16:48:22,674 - strands.telemetry.metrics - INFO - Creating Strands MetricsClient


I'll start by retrieving the KB context to get the database and catalog information, then proceed with the workflow.
Tool #1: retrieve_kb_context


2026-03-22 16:48:24,398 - metadata_query_agent.main - INFO - === retrieve_kb_context STARTED ===
2026-03-22 16:48:24,399 - metadata_query_agent.main - INFO - User query: How many unique admincodes exist?
2026-03-22 16:48:25,028 - metadata_query_agent.main - INFO - === retrieve_kb_context COMPLETED - 5122 tokens ===


...

The KB context clearly shows an `admin_codes` table with an `admin_code_id` column — "admincodes" is simply a concatenated form of "admin_codes". This is a straightforward match, so I'll proceed to execute the query.
Tool #3: execute_sql_query


2026-03-22 16:48:31,550 - metadata_query_agent.main - INFO - === execute_sql_query STARTED ===
2026-03-22 16:48:31,551 - metadata_query_agent.main - INFO - SQL query: SELECT COUNT(DISTINCT admin_code_id) AS unique_admincodes FROM admin_codes
2026-03-22 16:48:31,551 - metadata_query_agent.main - INFO - Database: normalized
2026-03-22 16:48:31,801 - metadata_query_agent.main - INFO - S3 output location: s3://semantic-layer-athena-results/metadata-query-results/
2026-03-22 16:48:31,801 - metadata_query_agent.main - INFO - Query type    : S3_TABLES_ICEBERG 
2026-03-22 16:48:31,802 - metadata_query_agent.main - INFO - QueryExecutionContext: {'Database': 'normalized', 'Catalog': 's3tablescatalog/semantic-layer-analytics-tables'}
2026-03-22 16:48:31,802 - metadata_query_agent.main - INFO - Workgroup     : semantic-layer-workgroup
2026-03-22 16:48:32,737 - metadata_query_agent.main - INFO - Query submitted: execution_id=2ccd8997-f22d-4936-af8d-bd4120911c7d
2026-03-22 16:50:11,690 - metadata_qu

I encountered a permissions error when trying to execute the query. The current principal (user/role) does not have the necessary privileges to access the `admin_codes` table in the `normalized` database under the `s3tablescatalog/semantic-layer-analytics-tables` catalog. 

To resolve this, please check with your data platform or cloud admin team to ensure the appropriate IAM or Lake Formation permissions are granted to query this

2026-03-22 16:50:14,748 - metadata_query_agent.main - INFO - Session 5fe452f0 — returning structured response: sql_query=False, rows=0, kb_sources=4


 resource.
✓ Evaluation complete!
  Generated 5 evaluation report(s)


## Aggregate Metrics

Analyze scores across all test cases and evaluators.

In [10]:
# Aggregate evaluation scores across all test cases
evaluation_data = []

if reports:
    n_cases = len(reports[0].cases)
    for j in range(n_cases):
        case_dict = reports[0].cases[j]
        metadata = case_dict.get('metadata') or {}
        query = case_dict.get('input', '')
        case_data = {
            'test_id': case_dict.get('name', f'case_{j}'),
            'category': metadata.get('category', 'unknown'),
            'query': query[:50] + '...' if len(query) > 50 else query,
        }
        for k, report in enumerate(reports):
            eval_name = evaluators[k].get_type_name()
            case_data[eval_name] = report.scores[j] if j < len(report.scores) else None
        evaluation_data.append(case_data)

df_evals = pd.DataFrame(evaluation_data)
evaluator_cols = [c for c in df_evals.columns if c not in ['test_id', 'category', 'query']]

print("=== EVALUATION SCORES BY TEST CASE ===\n")
print(df_evals.to_string(index=False))

print("\n=== AVERAGE SCORES BY EVALUATOR ===\n")
print(df_evals[evaluator_cols].mean().to_string())

print("\n=== SCORES BY CATEGORY ===\n")
print(df_evals.groupby('category')[evaluator_cols].mean().to_string())

=== EVALUATION SCORES BY TEST CASE ===

test_id          category                             query  HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  ToolParameterAccuracyEvaluator  GoalSuccessRateEvaluator
  test1 admin_codes_test1 How many unique admincodes exist?                   1.0                    1.0                             1.0                             1.0                       1.0

=== AVERAGE SCORES BY EVALUATOR ===

HelpfulnessEvaluator              1.0
FaithfulnessEvaluator             1.0
ToolSelectionAccuracyEvaluator    1.0
ToolParameterAccuracyEvaluator    1.0
GoalSuccessRateEvaluator          1.0

=== SCORES BY CATEGORY ===

                   HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  ToolParameterAccuracyEvaluator  GoalSuccessRateEvaluator
category                                                                                                                                                
admin_c

## Pass/Fail Analysis

Identify which test cases passed or failed each evaluator.

In [11]:
# Analyze pass/fail status for each evaluator
pass_fail_data = []

if reports:
    n_cases = len(reports[0].cases)
    for j in range(n_cases):
        case_dict = reports[0].cases[j]
        metadata = case_dict.get('metadata') or {}
        case_pf = {
            'test_id': case_dict.get('name', f'case_{j}'),
            'category': metadata.get('category', 'unknown'),
        }
        for k, report in enumerate(reports):
            eval_name = evaluators[k].get_type_name()
            case_pf[eval_name] = report.test_passes[j] if j < len(report.test_passes) else None
        pass_fail_data.append(case_pf)

df_pass_fail = pd.DataFrame(pass_fail_data)
pf_cols = [c for c in df_pass_fail.columns if c not in ['test_id', 'category']]

print("=== PASS/FAIL BY TEST CASE ===\n")
print(df_pass_fail.to_string(index=False))

print("\n=== PASS RATE BY EVALUATOR ===\n")
print(df_pass_fail[pf_cols].mean().to_string())

print("\n=== PASS RATE BY CATEGORY ===\n")
print(df_pass_fail.groupby('category')[pf_cols].mean().to_string())

=== PASS/FAIL BY TEST CASE ===

test_id          category  HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  ToolParameterAccuracyEvaluator  GoalSuccessRateEvaluator
  test1 admin_codes_test1                  True                   True                            True                            True                      True

=== PASS RATE BY EVALUATOR ===

HelpfulnessEvaluator              1.0
FaithfulnessEvaluator             1.0
ToolSelectionAccuracyEvaluator    1.0
ToolParameterAccuracyEvaluator    1.0
GoalSuccessRateEvaluator          1.0

=== PASS RATE BY CATEGORY ===

                   HelpfulnessEvaluator  FaithfulnessEvaluator  ToolSelectionAccuracyEvaluator  ToolParameterAccuracyEvaluator  GoalSuccessRateEvaluator
category                                                                                                                                                
admin_codes_test1                   1.0                    1.0                       

## Save Evaluation Results

Export detailed evaluation data for further analysis.

In [12]:
import os as _os
_os.makedirs('../data/eval/results', exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# # Save scores DataFrame
# scores_file = f'../data/eval/results/metadata_query_evaluation_scores_{timestamp}.csv'
# df_evals.to_csv(scores_file, index=False)
# print(f"✓ Scores saved to: {scores_file}")

# # Save pass/fail DataFrame
# pass_fail_file = f'../data/eval/results/metadata_query_evaluation_pass_fail_{timestamp}.csv'
# df_pass_fail.to_csv(pass_fail_file, index=False)
# print(f"✓ Pass/fail data saved to: {pass_fail_file}")

# Save detailed JSON — one entry per test case with all evaluator results
detailed_results = []
if reports:
    n_cases = len(reports[0].cases)
    for j in range(n_cases):
        case_dict = reports[0].cases[j]
        metadata = case_dict.get('metadata') or {}
        case_result = {
            'test_id': case_dict.get('name', f'case_{j}'),
            'category': metadata.get('category', 'unknown'),
            'query': case_dict.get('input', ''),
            'expected': case_dict.get('expected_output', ''),
            'evaluations': {},
        }
        for k, report in enumerate(reports):
            eval_name = evaluators[k].get_type_name()
            case_result['evaluations'][eval_name] = {
                'score': report.scores[j] if j < len(report.scores) else None,
                'test_pass': report.test_passes[j] if j < len(report.test_passes) else None,
                'reason': report.reasons[j] if j < len(report.reasons) else '',
            }
        detailed_results.append(case_result)

detailed_file = f'../data/eval/results/metadata_query_evaluation_detailed_{timestamp}.json'
with open(detailed_file, 'w') as f:
    json.dump(detailed_results, f, indent=2)
print(f"✓ Detailed results saved to: {detailed_file}")

print(f"\n{'='*70}")
print(f"✓ All evaluation results saved")
# print(f"  Scores:    {scores_file}")
# print(f"  Pass/Fail: {pass_fail_file}")
print(f"  Detailed:  {detailed_file}")

✓ Detailed results saved to: ../data/eval/results/metadata_query_evaluation_detailed_20260322_165123.json

✓ All evaluation results saved
  Detailed:  ../data/eval/results/metadata_query_evaluation_detailed_20260322_165123.json


## Display Detailed Report

View comprehensive evaluation metrics for each evaluator.

In [13]:
if reports:
    for k, report in enumerate(reports):
        eval_name = evaluators[k].get_type_name()
        print(f"\n{'='*70}")
        print(f"=== REPORT: {eval_name} (overall_score={report.overall_score:.2f}) ===\n")
        report.display(
            include_input=True,
            include_actual_output=True,
            include_expected_output=True,
        )
else:
    print("No reports generated")


=== REPORT: HelpfulnessEvaluator (overall_score=1.00) ===



╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                   Test Case Results                                    
┏━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ index ┃ name  ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_output ┃ expected_output ┃
┡━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ ▶ 0   │ test1 │ 1.00  │ ✅        │ ...    │ ...   │ ...           │ ...             │
└───────┴───────┴───────┴───────────┴────────┴───────┴───────────────┴─────────────────┘


=== REPORT: FaithfulnessEvaluator (overall_score=1.00) ===



╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                   Test Case Results                                    
┏━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ index ┃ name  ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_output ┃ expected_output ┃
┡━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ ▶ 0   │ test1 │ 1.00  │ ✅        │ ...    │ ...   │ ...           │ ...             │
└───────┴───────┴───────┴───────────┴────────┴───────┴───────────────┴─────────────────┘


=== REPORT: GoalSuccessRateEvaluator (overall_score=1.00) ===



╭───────────────────────────────────────────── 📊 Evaluation Report ──────────────────────────────────────────────╮
│ Overall Score: 1.00           Pass Rate: 1.0                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                   Test Case Results                                    
┏━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ index ┃ name  ┃ score ┃ test_pass ┃ reason ┃ input ┃ actual_output ┃ expected_output ┃
┡━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ ▶ 0   │ test1 │ 1.00  │ ✅        │ ...    │ ...   │ ...           │ ...             │
└───────┴───────┴───────┴───────────┴────────┴───────┴───────────────┴─────────────────┘

## Summary

### Agent Workflow
The Metadata Query Agent answers natural language questions by querying the Bedrock Knowledge Base for semantic metadata, resolving ambiguous terms, and executing SQL on Athena:

1. **`retrieve_kb_context(user_query)`** — fetches table/column metadata and routing info (database, catalog) from Bedrock KB via vector search
2. **`disambiguate_query_terms(user_query, kb_context)`** — resolves natural language terms to Athena table names using KB chunk metadata and synonym extraction; signals CLEAR, AMBIGUOUS, or UNKNOWN
3. **`execute_sql_query(sql_query, database_name, catalog_id)`** — submits SQL to Athena with the correct execution context for standard Glue or S3 Tables (Iceberg) catalogs, polls for completion, and returns structured results

### Multi-Dimensional Evaluation
- **HelpfulnessEvaluator**: Measures answer quality and user satisfaction (7-level scale)
- **FaithfulnessEvaluator**: Detects hallucinations — does the answer match the SQL results? (5-level scale)
- **ToolSelectionAccuracyEvaluator**: Validates correct tool ordering (KB → disambiguate → SQL)
- **ToolParameterAccuracyEvaluator**: Checks SQL correctness, database_name, and catalog_id values
- **GoalSuccessRateEvaluator**: Measures end-to-end goal achievement

### Required Environment Variables
- `EVAL_METADATA_QUERY_ID` — DynamoDB item `id` to use for eval runs (must exist in the metadata table)
- `SEMANTIC_RAG_KB_ID` — Bedrock Knowledge Base ID
- `ATHENA_WORKGROUP` — Athena workgroup
- `PROJECT_NAME` or `ATHENA_RESULTS_BUCKET` — S3 bucket for Athena results
- `AWS_REGION` — AWS region (default: `us-east-1`)

### Next Steps
- Review failed test cases to understand agent weaknesses
- Expand the dataset with multi-table joins and edge cases (ambiguous terms, unknown tables)
- Track metrics over time as the agent prompt or KB content evolves